In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
BenignIDS unified pipeline runner (BIDS.py)

Strict sequential discipline (fail-fast):
  0.4 → 2.1 → 4.1 → 4.2 → 4.3 → 5.1 → 7.2 → 8.0 → 8.1 → 9.6

Artefacts:
  - Splits: OUT_ROOT/splits/{X_*.parquet, y_*.parquet, manifest.json}
  - Payload seq: STAGE_ROOT/payload_seq_preproc/{payload_seq_{train,val,test}.npz, manifest.json}
  - Models: OUT_ROOT/models/{lgbm_baseline.txt, lgbm_bo.txt, lgbm_hpo.txt, cnn.keras}
  - Reports: OUT_ROOT/reports/*.json and *.png (charts)
"""

from __future__ import annotations

# =====================================================
# Section 0.1 — Imports
# =====================================================
import os
import sys
import json
import time
import uuid
import argparse
import warnings
import importlib
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Any, Optional, Callable
from functools import wraps
from contextlib import contextmanager


# =====================================================
# Section 0.2 — Minimal lazy imports (fail-fast at stage entry)
# =====================================================
def _maybe_import(name: str) -> Optional[Any]:
    try:
        return importlib.import_module(name)
    except ImportError:
        return None


np = _maybe_import("numpy")
pd = _maybe_import("pandas")
spm = _maybe_import("scipy.sparse")
skm = _maybe_import("sklearn")
lgb = _maybe_import("lightgbm")
skopt = _maybe_import("skopt")
tf = _maybe_import("tensorflow")
matplotlib = _maybe_import("matplotlib")


# =====================================================
# Section 0.3 — Sanity config (single source of truth)
# =====================================================
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "out"))
STAGE_ROOT = Path(os.environ.get("STAGE_ROOT", "staging"))
SPLITS_DIR = Path(os.environ.get("SPLITS_DIR", str(OUT_ROOT / "splits")))
RANDOM_STATE = int(os.environ.get("RANDOM_STATE", "42"))
TARGET_COL = os.environ.get("TARGET_COL", "label")

OUT_ROOT.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
STAGE_ROOT.mkdir(parents=True, exist_ok=True)


# =====================================================
# Section 0.4 — Utilities (fail-fast)
# =====================================================
def _require(cond: bool, msg: str) -> None:
    if not cond:
        raise RuntimeError(msg)


def _now_utc() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


@contextmanager
def _timer(stage: str):
    start = time.time()
    print(f"[{stage}] start @ {_now_utc()}", flush=True)
    yield
    print(f"[{stage}] done  in {time.time() - start:.2f}s", flush=True)


def _save_json(path: Path, obj: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    # ensure JSON-serialisable (notably numpy scalars)
    def _default(o):
        if np is not None:
            if isinstance(o, getattr(np, "integer", ())):
                return int(o)
            if isinstance(o, getattr(np, "floating", ())):
                return float(o)
        raise TypeError(f"Object of type {type(o).__name__} is not JSON serialisable")

    path.write_text(json.dumps(obj, indent=2, default=_default), encoding="utf-8")


def _load_json(path: Path) -> Dict[str, Any]:
    _require(path.exists(), f"Missing JSON: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def _load_parquet(path: Path) -> Any:
    _require(pd is not None, "pandas required")
    _require(path.exists(), f"Missing parquet: {path}")
    return pd.read_parquet(path)


def _to_dense_float32(X: Any) -> Any:
    _require(np is not None, "numpy required")
    if spm is not None and spm.issparse(X):
        return X.toarray().astype(np.float32, copy=False)
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32, copy=False)
    return np.asarray(X, dtype=np.float32)


def _to_lgb_matrix(X: Any) -> Any:
    """LightGBM accepts numpy or CSR; keep sparse if already sparse."""
    _require(np is not None, "numpy required")
    if spm is not None and spm.issparse(X):
        return X.tocsr()
    if hasattr(X, "select_dtypes"):
        return X.select_dtypes(include="number").to_numpy(dtype=np.float32, copy=False)
    return np.asarray(X, dtype=np.float32)


def _row_take(X: Any, idx: Any) -> Any:
    """Row-wise indexing that works for pandas, numpy, scipy sparse."""
    if hasattr(X, "iloc"):
        return X.iloc[idx]
    if spm is not None and spm.issparse(X):
        return X[idx]
    try:
        return X.take(idx, axis=0)
    except Exception:
        return X[idx]


def _ensure_binary_y(y: Any, ctx: str) -> Any:
    _require(np is not None, "numpy required")
    yv = np.asarray(y).astype(np.int32).ravel()
    u = set(np.unique(yv).tolist())
    _require(u.issubset({0, 1}) and len(u) == 2, f"[{ctx}] y must be binary with both classes present; got={u}")
    return yv


# =====================================================
# Section 0.5 — Decorators (dependency and file guards)
# =====================================================
def requires_libs(*libs: str):
    def decorator(func: Callable):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for lib in libs:
                _require(lib in globals() and globals()[lib] is not None, f"{lib} required for {func.__name__}")
            return func(*args, **kwargs)
        return wrapper
    return decorator


def requires_files(*paths: Path):
    def decorator(func: Callable):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for p in paths:
                _require(Path(p).exists(), f"Missing required file: {p}")
            return func(*args, **kwargs)
        return wrapper
    return decorator


# =====================================================
# Section 0.6 — Paths (reports/models)
# =====================================================
def _reports_dir() -> Path:
    p = OUT_ROOT / "reports"
    p.mkdir(parents=True, exist_ok=True)
    return p


def _models_dir() -> Path:
    p = OUT_ROOT / "models"
    p.mkdir(parents=True, exist_ok=True)
    return p


# =====================================================
# Section 0.7 — Plot helpers (saved PNGs + console messages)
# =====================================================
@requires_libs("np", "matplotlib", "skm")
def _plot_pr_curve(y_true: Any, y_score: Any, title: str, out_png: Path) -> None:
    import matplotlib.pyplot as plt
    from sklearn.metrics import precision_recall_curve, average_precision_score

    y_true = np.asarray(y_true).astype(np.int32).ravel()
    y_score = np.asarray(y_score).astype(np.float32).ravel()

    p, r, _ = precision_recall_curve(y_true, y_score)
    ap = float(average_precision_score(y_true, y_score))

    plt.figure(figsize=(6, 4))
    plt.plot(r, p, linewidth=2)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title} (AP={ap:.4f})")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=160)
    plt.close()
    print(f"[viz] PR curve → {out_png}", flush=True)


@requires_libs("np", "matplotlib", "skm")
def _plot_confusion(y_true: Any, y_score: Any, title: str, out_png: Path, thr: float = 0.5) -> None:
    import matplotlib.pyplot as plt
    from sklearn.metrics import confusion_matrix

    y_true = np.asarray(y_true).astype(np.int32).ravel()
    y_score = np.asarray(y_score).astype(np.float32).ravel()
    y_pred = (y_score >= float(thr)).astype(np.int32)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    plt.figure(figsize=(4.5, 4))
    plt.imshow(cm)
    plt.title(f"{title} (thr={thr:.2f})")
    plt.xticks([0, 1], ["0", "1"])
    plt.yticks([0, 1], ["0", "1"])
    plt.xlabel("Predicted")
    plt.ylabel("True")

    for (i, j), v in np.ndenumerate(cm):
        plt.text(j, i, str(int(v)), ha="center", va="center")

    plt.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=160)
    plt.close()
    print(f"[viz] Confusion matrix → {out_png}", flush=True)


@requires_libs("np", "matplotlib")
def _plot_ap_bar(models: Dict[str, float], out_png: Path) -> None:
    import matplotlib.pyplot as plt

    labels = list(models.keys())
    values = [float(models[k]) for k in labels]

    plt.figure(figsize=(7.5, 4.2))
    bars = plt.bar(labels, values)
    plt.ylim(0, 1.05)
    plt.ylabel("Average Precision (val)")
    plt.title("Model Performance Comparison (AP)")
    plt.grid(axis="y", alpha=0.3)
    for b, v in zip(bars, values):
        plt.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.3f}", ha="center", va="bottom")
    plt.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=160)
    plt.close()
    print(f"[viz] AP bar chart → {out_png}", flush=True)


@requires_libs("np", "matplotlib")
def _plot_ap_radar(models: Dict[str, float], out_png: Path) -> None:
    import matplotlib.pyplot as plt
    import numpy as _np

    labels = list(models.keys())
    vals = _np.array([float(models[k]) for k in labels], dtype=float)
    vals = _np.concatenate([vals, [vals[0]]])
    angles = _np.linspace(0, 2 * _np.pi, len(labels) + 1)

    plt.figure(figsize=(5.2, 5.2))
    ax = plt.subplot(111, polar=True)
    ax.plot(angles, vals, "o-", linewidth=2)
    ax.fill(angles, vals, alpha=0.25)
    ax.set_thetagrids(angles[:-1] * 180 / _np.pi, labels)
    ax.set_ylim(0, 1.05)
    ax.set_title("Model AP Radar Comparison")
    plt.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=160)
    plt.close()
    print(f"[viz] AP radar chart → {out_png}", flush=True)


# =====================================================
# Section 0.8 — Data classes
# =====================================================
@dataclass
class Features:
    X_train_payload: Any
    X_val_payload: Any
    X_test_payload: Any
    y_train: Any
    y_val: Any
    y_test: Any
    X_train_num: Any
    X_val_num: Any
    X_test_num: Any
    FEAT_KIND: str


# =====================================================
# Section 0.9 — Stage 0.4: split + persist parquet
# =====================================================
@requires_libs("pd", "np", "skm")
def split_and_persist_0_4(df: Any, target_col: str = TARGET_COL) -> None:
    """
    Persists:
      SPLITS_DIR/{X_train,X_val,X_test}.parquet
      SPLITS_DIR/{y_train,y_val,y_test}.parquet (column name == TARGET_COL)
      SPLITS_DIR/manifest.json
    """
    with _timer("0.4"):
        from sklearn.model_selection import train_test_split

        _require(target_col in df.columns, f"[0.4] Missing target col '{target_col}' in input dataframe")

        y = np.asarray(df[target_col]).astype(np.int32).ravel()
        u = set(np.unique(y).tolist())
        _require(u.issubset({0, 1}) and len(u) == 2, f"[0.4] Target must be binary with both classes; got={u}")

        X = df.drop(columns=[target_col])

        X_train, X_tmp, y_train, y_tmp = train_test_split(
            X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=RANDOM_STATE
        )

        # Persist
        X_train.to_parquet(SPLITS_DIR / "X_train.parquet", index=False)
        X_val.to_parquet(SPLITS_DIR / "X_val.parquet", index=False)
        X_test.to_parquet(SPLITS_DIR / "X_test.parquet", index=False)

        pd.DataFrame({TARGET_COL: y_train}).to_parquet(SPLITS_DIR / "y_train.parquet", index=False)
        pd.DataFrame({TARGET_COL: y_val}).to_parquet(SPLITS_DIR / "y_val.parquet", index=False)
        pd.DataFrame({TARGET_COL: y_test}).to_parquet(SPLITS_DIR / "y_test.parquet", index=False)

        split_id = str(uuid.uuid4())
        manifest = {
            "split_id": split_id,
            "created_utc": _now_utc(),
            "schema_version": 1,
            "target_col": TARGET_COL,
            "counts": {
                "train": {"0": int((y_train == 0).sum()), "1": int((y_train == 1).sum())},
                "val": {"0": int((y_val == 0).sum()), "1": int((y_val == 1).sum())},
                "test": {"0": int((y_test == 0).sum()), "1": int((y_test == 1).sum())},
            },
        }
        _save_json(SPLITS_DIR / "manifest.json", manifest)

        print(f"[0.4] splits persisted → {SPLITS_DIR}", flush=True)
        print(f"[0.4] split_id={split_id}", flush=True)
        print(f"[0.4] train: {manifest['counts']['train']}", flush=True)
        print(f"[0.4]   val: {manifest['counts']['val']}", flush=True)
        print(f"[0.4]  test: {manifest['counts']['test']}", flush=True)


# =====================================================
# Section 2.1 — Payload sequence preprocessing (byte-hist 256)
# =====================================================
@requires_libs("pd", "np", "spm")
@requires_files(SPLITS_DIR / "manifest.json", SPLITS_DIR / "X_train.parquet", SPLITS_DIR / "X_val.parquet", SPLITS_DIR / "X_test.parquet")
def preprocess_payload_2_1() -> None:
    """
    Persists:
      STAGE_ROOT/payload_seq_preproc/payload_seq_{train,val,test}.npz
      STAGE_ROOT/payload_seq_preproc/manifest.json
    """
    with _timer("2.1"):
        seq_dir = STAGE_ROOT / "payload_seq_preproc"
        seq_dir.mkdir(parents=True, exist_ok=True)

        sm = _load_json(SPLITS_DIR / "manifest.json")
        split_id = sm["split_id"]

        Xtr = _load_parquet(SPLITS_DIR / "X_train.parquet")
        Xva = _load_parquet(SPLITS_DIR / "X_val.parquet")
        Xte = _load_parquet(SPLITS_DIR / "X_test.parquet")

        payload_col = next((c for c in ["payload", "Payload", "PAYLOAD"] if c in Xtr.columns), None)

        def _byte_hist(series) -> Any:
            # series: pandas Series
            rows, cols, data = [], [], []
            series = series.astype(str)
            # IMPORTANT: enumerate over full length to keep row indices aligned
            for i, s in enumerate(series.tolist()):
                if s in ("nan", "None", ""):
                    continue
                b = s.encode("utf-8", errors="ignore")
                if not b:
                    continue
                hist = np.bincount(np.frombuffer(b, dtype=np.uint8), minlength=256)
                nz = np.flatnonzero(hist)
                if nz.size == 0:
                    continue
                rows.extend([i] * int(nz.size))
                cols.extend(nz.tolist())
                data.extend(hist[nz].astype(np.float32).tolist())
            return spm.csr_matrix((data, (rows, cols)), shape=(len(series), 256), dtype=np.float32)

        if payload_col is None:
            warnings.warn("[2.1] No payload column found; persisting zero histograms.", RuntimeWarning)
            P_tr = spm.csr_matrix((Xtr.shape[0], 256), dtype=np.float32)
            P_va = spm.csr_matrix((Xva.shape[0], 256), dtype=np.float32)
            P_te = spm.csr_matrix((Xte.shape[0], 256), dtype=np.float32)
            fallback = True
            note = "payload column missing; zero-filled histograms persisted"
        else:
            P_tr = _byte_hist(Xtr[payload_col])
            P_va = _byte_hist(Xva[payload_col])
            P_te = _byte_hist(Xte[payload_col])
            fallback = False
            note = None

        spm.save_npz(seq_dir / "payload_seq_train.npz", P_tr)
        spm.save_npz(seq_dir / "payload_seq_val.npz", P_va)
        spm.save_npz(seq_dir / "payload_seq_test.npz", P_te)

        print(f"[2.1] payload_seq_train.npz → {tuple(P_tr.shape)}", flush=True)
        print(f"[2.1] payload_seq_val.npz   → {tuple(P_va.shape)}", flush=True)
        print(f"[2.1] payload_seq_test.npz  → {tuple(P_te.shape)}", flush=True)

        manifest = {
            "split_id": split_id,
            "created_utc": _now_utc(),
            "schema_version": 1,
            "vocab": "byte_hist_256",
            "payload_col": payload_col,
            "fallback": fallback,
            "shape": {"train": list(P_tr.shape), "val": list(P_va.shape), "test": list(P_te.shape)},
        }
        if note:
            manifest["note"] = note

        _save_json(seq_dir / "manifest.json", manifest)
        print(f"[2.1] manifest → {seq_dir / 'manifest.json'}", flush=True)


# =====================================================
# Section 4.1 — Hydrate aligned features from persisted artefacts
# =====================================================
@requires_libs("pd", "np", "spm")
@requires_files(
    SPLITS_DIR / "manifest.json",
    SPLITS_DIR / "X_train.parquet", SPLITS_DIR / "X_val.parquet", SPLITS_DIR / "X_test.parquet",
    SPLITS_DIR / "y_train.parquet", SPLITS_DIR / "y_val.parquet", SPLITS_DIR / "y_test.parquet",
    STAGE_ROOT / "payload_seq_preproc" / "manifest.json",
    STAGE_ROOT / "payload_seq_preproc" / "payload_seq_train.npz",
    STAGE_ROOT / "payload_seq_preproc" / "payload_seq_val.npz",
    STAGE_ROOT / "payload_seq_preproc" / "payload_seq_test.npz",
)
def hydrate_features_4_1() -> Features:
    """
    Guarantees alignment by construction: features are loaded from the same persisted split artefacts.
    """
    with _timer("4.1"):
        # payload
        seq_dir = STAGE_ROOT / "payload_seq_preproc"
        X_train_payload = spm.load_npz(seq_dir / "payload_seq_train.npz")
        X_val_payload = spm.load_npz(seq_dir / "payload_seq_val.npz")
        X_test_payload = spm.load_npz(seq_dir / "payload_seq_test.npz")

        # y
        y_train = _load_parquet(SPLITS_DIR / "y_train.parquet")[TARGET_COL].to_numpy(dtype=np.int32)
        y_val = _load_parquet(SPLITS_DIR / "y_val.parquet")[TARGET_COL].to_numpy(dtype=np.int32)
        y_test = _load_parquet(SPLITS_DIR / "y_test.parquet")[TARGET_COL].to_numpy(dtype=np.int32)

        # fail-fast: ensure lengths match for payload
        _require(X_train_payload.shape[0] == len(y_train), "[4.1] payload train rows != y_train")
        _require(X_val_payload.shape[0] == len(y_val), "[4.1] payload val rows != y_val")
        _require(X_test_payload.shape[0] == len(y_test), "[4.1] payload test rows != y_test")

        # numeric (strict: numeric-only)
        X_train_df = _load_parquet(SPLITS_DIR / "X_train.parquet")
        X_val_df = _load_parquet(SPLITS_DIR / "X_val.parquet")
        X_test_df = _load_parquet(SPLITS_DIR / "X_test.parquet")

        X_train_num = X_train_df.select_dtypes(include="number").to_numpy(dtype=np.float32, copy=False)
        X_val_num = X_val_df.select_dtypes(include="number").to_numpy(dtype=np.float32, copy=False)
        X_test_num = X_test_df.select_dtypes(include="number").to_numpy(dtype=np.float32, copy=False)

        _require(X_train_num.shape[0] == len(y_train), "[4.1] numeric train rows != y_train")
        _require(X_val_num.shape[0] == len(y_val), "[4.1] numeric val rows != y_val")
        _require(X_test_num.shape[0] == len(y_test), "[4.1] numeric test rows != y_test")

        # binary class presence (guarding later model training)
        _ensure_binary_y(y_train, "4.1/train")
        _ensure_binary_y(y_val, "4.1/val")
        _ensure_binary_y(y_test, "4.1/test")

        print(
            f"[4.1] numeric: {tuple(X_train_num.shape)} → {tuple(X_val_num.shape)} | "
            f"payload: {tuple(X_train_payload.shape)} → {tuple(X_val_payload.shape)}",
            flush=True
        )

        return Features(
            X_train_payload=X_train_payload,
            X_val_payload=X_val_payload,
            X_test_payload=X_test_payload,
            y_train=y_train,
            y_val=y_val,
            y_test=y_test,
            X_train_num=X_train_num,
            X_val_num=X_val_num,
            X_test_num=X_test_num,
            FEAT_KIND="numeric(select_dtypes)+payload(byte_hist_256)",
        )


# =====================================================
# Section 4.2 — Baseline LightGBM (numeric features)
# =====================================================
@requires_libs("lgb", "np", "skm")
def train_baseline_lgbm_4_2(feat: Features) -> Dict[str, Any]:
    with _timer("4.2"):
        from sklearn.metrics import average_precision_score

        Xtr = _to_lgb_matrix(feat.X_train_num)
        Xva = _to_lgb_matrix(feat.X_val_num)

        ytr = _ensure_binary_y(feat.y_train, "4.2/train")
        yva = _ensure_binary_y(feat.y_val, "4.2/val")

        _require(Xtr.shape[0] == len(ytr), "[4.2] X_train_num rows != y_train")
        _require(Xva.shape[0] == len(yva), "[4.2] X_val_num rows != y_val")

        dtrain = lgb.Dataset(Xtr, label=ytr, free_raw_data=True)
        dval = lgb.Dataset(Xva, label=yva, reference=dtrain, free_raw_data=True)

        params = {
            "objective": "binary",
            "metric": ["average_precision", "binary_logloss"],
            "boosting_type": "gbdt",
            "num_leaves": 31,
            "learning_rate": 0.05,
            "feature_fraction": 0.9,
            "verbosity": -1,
            "seed": RANDOM_STATE,
        }

        booster = lgb.train(
            params,
            dtrain,
            valid_sets=[dval],
            num_boost_round=400,
            callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(50)],
        )

        yva_pred = booster.predict(Xva, num_iteration=booster.best_iteration)
        ap = float(average_precision_score(yva, yva_pred))

        model_path = _models_dir() / "lgbm_baseline.txt"
        booster.save_model(str(model_path))

        rep = {
            "status": "ok",
            "model": "lgbm_baseline",
            "features": "numeric",
            "ap_val": ap,
            "best_iteration": int(booster.best_iteration),
            "model_path": str(model_path),
            "timestamp_utc": _now_utc(),
        }
        _save_json(_reports_dir() / "lgbm_baseline.json", rep)

        # visuals
        _plot_pr_curve(yva, yva_pred, "LightGBM Baseline (val)", _reports_dir() / "pr_lgbm_baseline.png")
        _plot_confusion(yva, yva_pred, "LightGBM Baseline (val)", _reports_dir() / "cm_lgbm_baseline.png", thr=0.5)

        print(f"[4.2] baseline AP={ap:.4f}  best_it={booster.best_iteration}", flush=True)
        return {"ap_val": ap, "model_path": str(model_path)}


# =====================================================
# Section 4.3 — Bayesian optimisation (LightGBM, numeric features)
# =====================================================
@requires_libs("lgb", "np", "skopt", "skm")
def optimize_lgbm_4_3(feat: Features) -> Dict[str, Any]:
    with _timer("4.3"):
        from skopt import gp_minimize
        from skopt.space import Real, Integer
        from skopt.utils import use_named_args
        from skopt.callbacks import CheckpointSaver
        from sklearn.metrics import average_precision_score

        Xtr = _to_lgb_matrix(feat.X_train_num)
        Xva = _to_lgb_matrix(feat.X_val_num)

        ytr = _ensure_binary_y(feat.y_train, "4.3/train")
        yva = _ensure_binary_y(feat.y_val, "4.3/val")

        _require(Xtr.shape[0] == len(ytr), "[4.3] X_train_num rows != y_train")
        _require(Xva.shape[0] == len(yva), "[4.3] X_val_num rows != y_val")

        space = [
            Integer(20, 120, name="num_leaves"),
            Real(0.005, 0.2, name="learning_rate", prior="log-uniform"),
            Integer(20, 250, name="min_data_in_leaf"),
            Real(0.6, 1.0, name="feature_fraction"),
        ]

        @use_named_args(space)
        def objective(**p):
            params = {
                "objective": "binary",
                "metric": "average_precision",
                "boosting_type": "gbdt",
                "num_leaves": int(p["num_leaves"]),
                "learning_rate": float(p["learning_rate"]),
                "min_data_in_leaf": int(p["min_data_in_leaf"]),
                "feature_fraction": float(p["feature_fraction"]),
                "verbosity": -1,
                "seed": RANDOM_STATE,
            }
            dtrain = lgb.Dataset(Xtr, label=ytr, free_raw_data=True)
            dval = lgb.Dataset(Xva, label=yva, reference=dtrain, free_raw_data=True)
            booster = lgb.train(
                params,
                dtrain,
                valid_sets=[dval],
                num_boost_round=400,
                callbacks=[lgb.early_stopping(30, verbose=False)],
            )
            y_pred = booster.predict(Xva, num_iteration=booster.best_iteration)
            ap = float(average_precision_score(yva, y_pred))
            return -ap

        reports_dir = _reports_dir()
        ckpt_path = reports_dir / "lgbm_bo.ckpt.pkl"
        checkpoint_cb = CheckpointSaver(str(ckpt_path), compress=9, store_objective=False)

        res = gp_minimize(
            objective,
            dimensions=space,
            n_calls=int(os.environ.get("BO_N_CALLS", "25")),
            n_initial_points=int(os.environ.get("BO_N_INIT", "8")),
            random_state=RANDOM_STATE,
            callback=[checkpoint_cb],
            verbose=False,
        )

        best_params = {
            "num_leaves": int(res.x[0]),
            "learning_rate": float(res.x[1]),
            "min_data_in_leaf": int(res.x[2]),
            "feature_fraction": float(res.x[3]),
        }

        train_params = {
            "objective": "binary",
            "metric": ["average_precision", "binary_logloss"],
            "boosting_type": "gbdt",
            "verbosity": -1,
            "seed": RANDOM_STATE,
            **best_params,
        }

        dtrain = lgb.Dataset(Xtr, label=ytr, free_raw_data=True)
        dval = lgb.Dataset(Xva, label=yva, reference=dtrain, free_raw_data=True)

        booster = lgb.train(
            train_params,
            dtrain,
            valid_sets=[dval],
            num_boost_round=600,
            callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(50)],
        )

        yva_pred = booster.predict(Xva, num_iteration=booster.best_iteration)
        ap = float(average_precision_score(yva, yva_pred))

        model_path = _models_dir() / "lgbm_bo.txt"
        booster.save_model(str(model_path))

        rep = {
            "status": "ok",
            "model": "lgbm_bo",
            "features": "numeric",
            "ap_val": ap,
            "best_iteration": int(booster.best_iteration),
            "best_params": best_params,
            "model_path": str(model_path),
            "timestamp_utc": _now_utc(),
        }
        _save_json(reports_dir / "lgbm_bo.json", rep)

        # BO visuals (guarded: skopt may not store models depending on checkpoint config)
        try:
            import matplotlib.pyplot as plt
            from skopt.plots import plot_convergence
            plt.figure(figsize=(6, 4))
            plot_convergence(res)
            plt.title("LightGBM BO Convergence")
            plt.grid(alpha=0.3)
            plt.tight_layout()
            out_png = reports_dir / "bo_convergence.png"
            plt.savefig(out_png, dpi=160)
            plt.close()
            print(f"[viz] BO convergence → {out_png}", flush=True)
        except Exception as e:
            print(f"[viz] BO convergence skipped: {e}", flush=True)

        _plot_pr_curve(yva, yva_pred, "LightGBM BO (val)", reports_dir / "pr_lgbm_bo.png")
        _plot_confusion(yva, yva_pred, "LightGBM BO (val)", reports_dir / "cm_lgbm_bo.png", thr=0.5)

        print(f"[4.3] BO AP={ap:.4f}  best={best_params}", flush=True)
        return {"ap_val": ap, "model_path": str(model_path), "best_params": best_params}


# =====================================================
# Section 5.1 — HPO (RandomisedSearchCV on LightGBM, numeric features)
# =====================================================
@requires_libs("lgb", "np", "skm")
def hpo_lgbm_5_1(feat: Features) -> Dict[str, Any]:
    """
    Produces:
      OUT_ROOT/models/lgbm_hpo.txt
      OUT_ROOT/reports/lgbm_hpo.json
      + PR/CM charts
    """
    with _timer("5.1"):
        from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
        from sklearn.metrics import average_precision_score, make_scorer
        from lightgbm import LGBMClassifier

        Xtr = _to_dense_float32(feat.X_train_num)
        Xva = _to_dense_float32(feat.X_val_num)

        ytr = _ensure_binary_y(feat.y_train, "5.1/train")
        yva = _ensure_binary_y(feat.y_val, "5.1/val")

        _require(Xtr.shape[0] == len(ytr), "[5.1] X_train_num rows != y_train")
        _require(Xva.shape[0] == len(yva), "[5.1] X_val_num rows != y_val")

        # search space (kept realistic for tabular IDS)
        param_dist = {
            "num_leaves": np.arange(24, 128, dtype=int),
            "learning_rate": np.logspace(-3, -1, 60),
            "min_child_samples": np.arange(20, 260, dtype=int),
            "subsample": np.linspace(0.6, 1.0, 21),
            "colsample_bytree": np.linspace(0.6, 1.0, 21),
            "reg_alpha": np.logspace(-4, 0, 40),
            "reg_lambda": np.logspace(-4, 0, 40),
        }

        base = LGBMClassifier(
            objective="binary",
            n_estimators=800,
            n_jobs=int(os.environ.get("LGBM_N_JOBS", "-1")),
            random_state=RANDOM_STATE,
        )

        cv = StratifiedKFold(n_splits=int(os.environ.get("HPO_CV_SPLITS", "3")), shuffle=True, random_state=RANDOM_STATE)
        scorer = make_scorer(average_precision_score, needs_proba=True)

        n_iter = int(os.environ.get("HPO_N_ITER", "20"))
        rs = RandomizedSearchCV(
            estimator=base,
            param_distributions=param_dist,
            n_iter=n_iter,
            scoring=scorer,
            cv=cv,
            random_state=RANDOM_STATE,
            verbose=0,
            n_jobs=int(os.environ.get("HPO_N_JOBS", "-1")),
        )

        rs.fit(Xtr, ytr)

        best = rs.best_estimator_
        yva_pred = best.predict_proba(Xva)[:, 1]
        ap = float(average_precision_score(yva, yva_pred))

        model_path = _models_dir() / "lgbm_hpo.txt"
        best.booster_.save_model(str(model_path))

        rep = {
            "status": "ok",
            "model": "lgbm_hpo",
            "features": "numeric",
            "ap_val": ap,
            "best_params": dict(rs.best_params_),
            "cv_best_score": float(rs.best_score_),
            "model_path": str(model_path),
            "timestamp_utc": _now_utc(),
        }
        _save_json(_reports_dir() / "lgbm_hpo.json", rep)

        _plot_pr_curve(yva, yva_pred, "LightGBM HPO (val)", _reports_dir() / "pr_lgbm_hpo.png")
        _plot_confusion(yva, yva_pred, "LightGBM HPO (val)", _reports_dir() / "cm_lgbm_hpo.png", thr=0.5)

        print(f"[5.1] HPO AP={ap:.4f}  cv_best={rs.best_score_:.4f}", flush=True)
        return {"ap_val": ap, "model_path": str(model_path), "best_params": dict(rs.best_params_)}


# =====================================================
# Section 7.2 — CNN training (payload byte-histograms)
# =====================================================
@requires_libs("tf", "np", "skm")
def train_cnn_7_2(feat: Features) -> Dict[str, Any]:
    """
    CNN over 16x16 reshaped byte-hist (256).
    Produces:
      OUT_ROOT/models/cnn.keras
      OUT_ROOT/reports/cnn.json
      + PR/CM charts
    """
    with _timer("7.2"):
        from tensorflow import keras
        from sklearn.metrics import average_precision_score

        Xtr_sp = feat.X_train_payload
        Xva_sp = feat.X_val_payload

        ytr = _ensure_binary_y(feat.y_train, "7.2/train")
        yva = _ensure_binary_y(feat.y_val, "7.2/val")

        _require(Xtr_sp.shape[1] == 256, f"[7.2] Expected 256 payload features; got {Xtr_sp.shape[1]}")
        _require(Xva_sp.shape[1] == 256, f"[7.2] Expected 256 payload features; got {Xva_sp.shape[1]}")

        Xtr = _to_dense_float32(Xtr_sp)
        Xva = _to_dense_float32(Xva_sp)

        # reshape to image-like
        Xtr = Xtr.reshape((-1, 16, 16, 1))
        Xva = Xva.reshape((-1, 16, 16, 1))

        model = keras.Sequential([
            keras.layers.Input(shape=(16, 16, 1)),
            keras.layers.Conv2D(16, (3, 3), padding="same", activation="relu"),
            keras.layers.MaxPool2D((2, 2)),
            keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
            keras.layers.MaxPool2D((2, 2)),
            keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
            keras.layers.GlobalAveragePooling2D(),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dropout(0.25),
            keras.layers.Dense(1, activation="sigmoid"),
        ])

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=float(os.environ.get("CNN_LR", "1e-3"))),
            loss="binary_crossentropy",
            metrics=[keras.metrics.AUC(name="auc"), keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")],
        )

        cb = [
            keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True),
        ]

        epochs = int(os.environ.get("CNN_EPOCHS", "20"))
        batch = int(os.environ.get("CNN_BATCH", "256"))

        history = model.fit(
            Xtr, ytr,
            validation_data=(Xva, yva),
            epochs=epochs,
            batch_size=batch,
            callbacks=cb,
            verbose=0,
        )

        yva_pred = model.predict(Xva, verbose=0).ravel()
        ap = float(average_precision_score(yva, yva_pred))

        model_path = _models_dir() / "cnn.keras"
        model.save(model_path)

        rep = {
            "status": "ok",
            "model": "cnn_16x16_hist",
            "features": "payload(byte_hist_256)",
            "ap_val": ap,
            "epochs_ran": int(len(history.history.get("loss", []))),
            "model_path": str(model_path),
            "timestamp_utc": _now_utc(),
        }
        _save_json(_reports_dir() / "cnn.json", rep)

        _plot_pr_curve(yva, yva_pred, "CNN (val)", _reports_dir() / "pr_cnn.png")
        _plot_confusion(yva, yva_pred, "CNN (val)", _reports_dir() / "cm_cnn.png", thr=0.5)

        # training curves
        try:
            import matplotlib.pyplot as plt
            hist = history.history
            if "loss" in hist and "val_loss" in hist:
                plt.figure(figsize=(6, 4))
                plt.plot(hist["loss"], label="train_loss")
                plt.plot(hist["val_loss"], label="val_loss")
                plt.title("CNN Loss Curves")
                plt.xlabel("epoch")
                plt.ylabel("loss")
                plt.grid(alpha=0.3)
                plt.legend()
                plt.tight_layout()
                out_png = _reports_dir() / "cnn_loss.png"
                plt.savefig(out_png, dpi=160)
                plt.close()
                print(f"[viz] CNN loss → {out_png}", flush=True)

            if "auc" in hist and "val_auc" in hist:
                plt.figure(figsize=(6, 4))
                plt.plot(hist["auc"], label="train_auc")
                plt.plot(hist["val_auc"], label="val_auc")
                plt.title("CNN AUC Curves")
                plt.xlabel("epoch")
                plt.ylabel("auc")
                plt.grid(alpha=0.3)
                plt.legend()
                plt.tight_layout()
                out_png = _reports_dir() / "cnn_auc.png"
                plt.savefig(out_png, dpi=160)
                plt.close()
                print(f"[viz] CNN AUC → {out_png}", flush=True)
        except Exception as e:
            print(f"[viz] CNN training curves skipped: {e}", flush=True)

        print(f"[7.2] CNN AP={ap:.4f}", flush=True)
        return {"ap_val": ap, "model_path": str(model_path)}


# =====================================================
# Section 8.0 — Manifest (reports/models inventory)
# =====================================================
def generate_report_8_0() -> None:
    with _timer("8.0"):
        rep_dir = _reports_dir()
        mod_dir = _models_dir()

        artefacts = {
            "timestamp_utc": _now_utc(),
            "reports": sorted([p.name for p in rep_dir.glob("*.json")]) + sorted([p.name for p in rep_dir.glob("*.png")]),
            "models": sorted([p.name for p in mod_dir.glob("*")]),
        }
        _save_json(OUT_ROOT / "manifest.json", artefacts)
        print(f"[8.0] manifest → {OUT_ROOT / 'manifest.json'}", flush=True)


# =====================================================
# Section 8.1 — Console report + comparison charts (Baseline vs BO vs HPO vs CNN)
# =====================================================
@requires_libs("np", "matplotlib")
def generate_extended_report_8_1() -> None:
    with _timer("8.1"):
        rep_dir = _reports_dir()

        # Load split counts
        split_counts = None
        try:
            m = _load_json(SPLITS_DIR / "manifest.json")
            split_counts = m.get("counts")
        except Exception:
            split_counts = None

        def _ap_from(name: str) -> Optional[float]:
            p = rep_dir / name
            if not p.exists():
                return None
            try:
                r = _load_json(p)
                v = r.get("ap_val", None)
                return float(v) if v is not None else None
            except Exception:
                return None

        ap_base = _ap_from("lgbm_baseline.json")
        ap_bo = _ap_from("lgbm_bo.json")
        ap_hpo = _ap_from("lgbm_hpo.json")
        ap_cnn = _ap_from("cnn.json")

        # Console report (always)
        print("\n================== IDS REPORT ==================", flush=True)
        if split_counts:
            print(f"train: {split_counts.get('train')}", flush=True)
            print(f"  val: {split_counts.get('val')}", flush=True)
            print(f" test: {split_counts.get('test')}", flush=True)
        else:
            print("[8.1] split counts unavailable (missing splits manifest).", flush=True)

        print(f"Baseline AP={ap_base}", flush=True)
        print(f"BO       AP={ap_bo}", flush=True)
        print(f"HPO      AP={ap_hpo}", flush=True)
        print(f"CNN      AP={ap_cnn}", flush=True)
        print("================================================\n", flush=True)

        # Charts (only plot available APs)
        models = {}
        if ap_base is not None: models["Baseline"] = float(ap_base)
        if ap_bo is not None: models["BayesOpt"] = float(ap_bo)
        if ap_hpo is not None: models["HPO"] = float(ap_hpo)
        if ap_cnn is not None: models["CNN"] = float(ap_cnn)

        _require(len(models) >= 2, "[8.1] Need at least two model reports to compare (expected baseline/bo/hpo/cnn).")

        _plot_ap_bar(models, rep_dir / "comparison_ap.png")
        _plot_ap_radar(models, rep_dir / "comparison_radar.png")


# =====================================================
# Section 9.6 — Champion selection (max AP)
# =====================================================
@requires_libs("np")
def select_champion_9_6() -> None:
    with _timer("9.6"):
        rep_dir = _reports_dir()
        mod_dir = _models_dir()

        candidates = []

        def _add(model: str, report_json: str, default_model_file: Optional[str] = None):
            p = rep_dir / report_json
            if not p.exists():
                return
            r = _load_json(p)
            ap = r.get("ap_val", None)
            if ap is None:
                return
            ap = float(ap)
            if not np.isfinite(ap):
                return

            path = r.get("model_path", None)
            if path is None and default_model_file is not None:
                mp = mod_dir / default_model_file
                path = str(mp) if mp.exists() else str(p)

            candidates.append({
                "model": model,
                "ap_val": ap,
                "path": str(path) if path is not None else str(p),
            })

        _add("lgbm_baseline", "lgbm_baseline.json", "lgbm_baseline.txt")
        _add("lgbm_bo", "lgbm_bo.json", "lgbm_bo.txt")
        _add("lgbm_hpo", "lgbm_hpo.json", "lgbm_hpo.txt")
        _add("cnn_16x16_hist", "cnn.json", "cnn.keras")

        if candidates:
            champ = max(candidates, key=lambda x: x["ap_val"])
            obj = {
                "status": "ready",
                "champion": champ,
                "candidates": sorted(candidates, key=lambda x: x["ap_val"], reverse=True),
                "timestamp_utc": _now_utc(),
            }
            print(f"[9.6] Champion → {champ['model']} (AP={champ['ap_val']:.4f})", flush=True)
        else:
            obj = {
                "status": "pending",
                "reason": "No valid model scores available for champion selection.",
                "required_next_steps": ["Run 4.2, 4.3, 5.1, 7.2 to produce comparable metrics."],
                "timestamp_utc": _now_utc(),
            }
            print("[9.6] Champion pending — no valid models found", flush=True)

        _save_json(rep_dir / "champion.json", obj)
        _save_json(rep_dir / "selection_summary.json", obj)
        (rep_dir / "selection_summary.txt").write_text(json.dumps(obj, indent=2), encoding="utf-8")
        print("[9.6] champion.json + selection_summary.{json,txt} written", flush=True)


# =====================================================
# Section 10.0 — CLI + sequential runner
# =====================================================
def main():
    parser = argparse.ArgumentParser(
        description="BenignIDS unified pipeline runner",
        add_help=True,
        conflict_handler="resolve",
    )
    parser.add_argument(
        "--stage",
        type=str,
        default="all",
        choices=["0.4", "2.1", "4.1", "4.2", "4.3", "5.1", "7.2", "8.0", "8.1", "9.6", "all"],
        help="Pipeline stage to run",
    )
    parser.add_argument("--csv", type=str, default=None, help="CSV path for stage 0.4")
    parser.add_argument("--target", type=str, default=TARGET_COL, help="Target column name")
    parser.add_argument("--f", type=str, default=None, help=argparse.SUPPRESS)  # Jupyter noise

    args, _ = parser.parse_known_args()

    def _require_splits():
        req = [
            SPLITS_DIR / "manifest.json",
            SPLITS_DIR / "X_train.parquet", SPLITS_DIR / "X_val.parquet", SPLITS_DIR / "X_test.parquet",
            SPLITS_DIR / "y_train.parquet", SPLITS_DIR / "y_val.parquet", SPLITS_DIR / "y_test.parquet",
        ]
        for p in req:
            _require(p.exists(), f"Missing prerequisite: {p}")

    def _require_payload_preproc():
        req = [
            STAGE_ROOT / "payload_seq_preproc" / "manifest.json",
            STAGE_ROOT / "payload_seq_preproc" / "payload_seq_train.npz",
            STAGE_ROOT / "payload_seq_preproc" / "payload_seq_val.npz",
            STAGE_ROOT / "payload_seq_preproc" / "payload_seq_test.npz",
        ]
        for p in req:
            _require(p.exists(), f"Missing prerequisite: {p}")

    with _timer("Pipeline"):
        if args.stage == "all":
            print(">>> BIDS.py — full pipeline start", flush=True)

            # 0.4 (only if CSV provided; otherwise enforce existing splits)
            if args.csv is not None:
                _require(pd is not None, "pandas required")
                df = pd.read_csv(args.csv)
                _require(args.target in df.columns, f"Target '{args.target}' not in CSV")
                split_and_persist_0_4(df, args.target)
            else:
                _require_splits()
                print(">>> resume: existing splits detected", flush=True)

            # strict sequence
            preprocess_payload_2_1()
            _require_payload_preproc()

            feat = hydrate_features_4_1()
            train_baseline_lgbm_4_2(feat)
            optimize_lgbm_4_3(feat)
            hpo_lgbm_5_1(feat)
            train_cnn_7_2(feat)

            generate_report_8_0()
            generate_extended_report_8_1()
            select_champion_9_6()

            print(">>> BIDS.py — full pipeline complete", flush=True)
            return

        # single-stage runner (still disciplined)
        if args.stage == "0.4":
            _require(args.csv is not None, "--csv required for stage 0.4")
            _require(pd is not None, "pandas required")
            df = pd.read_csv(args.csv)
            _require(args.target in df.columns, f"Target '{args.target}' not in CSV")
            split_and_persist_0_4(df, args.target)
            return

        if args.stage == "2.1":
            _require_splits()
            preprocess_payload_2_1()
            return

        if args.stage == "4.1":
            _require_splits()
            _require_payload_preproc()
            hydrate_features_4_1()
            return

        if args.stage in ("4.2", "4.3", "5.1", "7.2"):
            _require_splits()
            _require_payload_preproc()
            feat = hydrate_features_4_1()
            if args.stage == "4.2":
                train_baseline_lgbm_4_2(feat)
            elif args.stage == "4.3":
                optimize_lgbm_4_3(feat)
            elif args.stage == "5.1":
                hpo_lgbm_5_1(feat)
            elif args.stage == "7.2":
                train_cnn_7_2(feat)
            return

        if args.stage == "8.0":
            generate_report_8_0()
            return

        if args.stage == "8.1":
            generate_extended_report_8_1()
            return

        if args.stage == "9.6":
            select_champion_9_6()
            return

        _require(False, f"Invalid stage: {args.stage}")


if __name__ == "__main__":
    main()


[Pipeline] start @ 2025-12-14T19:04:02Z


AssertionError: [all] Missing persisted splits: ['out/splits/manifest.json', 'out/splits/X_train.parquet', 'out/splits/X_val.parquet', 'out/splits/y_train.parquet', 'out/splits/y_val.parquet']. Run --stage 0.4 with --csv first.